In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import polars as pl

from config import (
    RAW_DATA_DIR,
    SAML_D_DATASET_HANDLE,
    SAML_D_FILE_NAME,
    SAML_D_INTERIM_PATH
)

from src.data.acquisition import KaggleDatasetDownloader
from src.data.loader import SAMLDDataLoader
from src.data.validation import SAMLDDataValidator
from src.data.preprocessing import SAMLDPreprocessor

In [2]:
### data acquisition
# create a downloader instance
downloader = KaggleDatasetDownloader(
    dataset_handle = SAML_D_DATASET_HANDLE,
    file_name = SAML_D_FILE_NAME,
    destination_dir = RAW_DATA_DIR
)

# download the raw SAML-D dataset
raw_data_path = downloader.download(
    force = False
)

raw_data_path

PosixPath('/Users/gorkemy/Desktop/Projects/Graph-based Financial Crime/data/raw/SAML-D.csv')

In [3]:
### data inspection
# create a loader instance
loader = SAMLDDataLoader(
    raw_data_path = raw_data_path
)

# create lazy query plan
raw_lazy_frame = loader.scan_raw()

# display LazyFrame schema
display(raw_lazy_frame.collect_schema())

# load a LazyFrame sample
raw_sample = loader.load_raw_sample(
    n_rows = 10_000
)

# inspect first 10 rows
display(raw_sample.head(10))

# inspecting the primary label
display(
    raw_sample
    .group_by('Is_laundering')
    .len()
)

Schema([('Time', String),
        ('Date', String),
        ('Sender_account', String),
        ('Receiver_account', String),
        ('Amount', Float64),
        ('Payment_currency', String),
        ('Received_currency', String),
        ('Sender_bank_location', String),
        ('Receiver_bank_location', String),
        ('Payment_type', String),
        ('Is_laundering', Int8),
        ('Laundering_type', String)])

Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
str,str,str,str,f64,str,str,str,str,str,i8,str
"""10:35:19""","""2022-10-07""","""8724731955""","""2769355426""",1459.15,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,"""Normal_Cash_Deposits"""
"""10:35:20""","""2022-10-07""","""1491989064""","""8401255335""",6019.64,"""UK pounds""","""Dirham""","""UK""","""UAE""","""Cross-border""",0,"""Normal_Fan_Out"""
"""10:35:20""","""2022-10-07""","""287305149""","""4404767002""",14328.44,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,"""Normal_Small_Fan_Out"""
"""10:35:21""","""2022-10-07""","""5376652437""","""9600420220""",11895.0,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Fan_In"""
"""10:35:21""","""2022-10-07""","""9614186178""","""3803336972""",115.25,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,"""Normal_Cash_Deposits"""
"""10:35:21""","""2022-10-07""","""8974559268""","""3143547511""",5130.99,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Group"""
"""10:35:23""","""2022-10-07""","""980191499""","""8577635959""",12176.52,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Small_Fan_Out"""
"""10:35:23""","""2022-10-07""","""8057793308""","""9350896213""",56.9,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Credit card""",0,"""Normal_Small_Fan_Out"""
"""10:35:26""","""2022-10-07""","""6116657264""","""656192169""",4738.45,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,"""Normal_Fan_Out"""


Is_laundering,len
i8,u32
1,13
0,9987


In [4]:
### data validation
# raw data content validation
raw_content_summary = SAMLDDataValidator.validate_content(
    lazy_frame = raw_lazy_frame
)

raw_content_summary_frame = pl.DataFrame(
    [raw_content_summary]
)

with pl.Config(
    tbl_rows=-1
):
    display(
    raw_content_summary_frame
    .transpose(
        include_header = True,
        header_name = 'metric',
        column_names = ['value']
    )
)

metric,value
str,i64
"""row_count""",9504852
"""time_null_count""",0
"""date_null_count""",0
"""sender_account_null_count""",0
"""receiver_account_null_count""",0
"""amount_null_count""",0
"""payment_currency_null_count""",0
"""received_currency_null_count""",0
"""sender_bank_location_null_coun…",0


In [5]:
### data acquisition after Parquet conversion
# create preprocessor instance
preprocessor = SAMLDPreprocessor(
    raw_data_path = raw_data_path,
    interim_data_path = SAML_D_INTERIM_PATH
)

# convert raw dataset into parquet and write to interim data path
interim_data_path = preprocessor.convert_to_parquet(
    overwrite = False
)

interim_data_path

PosixPath('/Users/gorkemy/Desktop/Projects/Graph-based Financial Crime/data/interim/saml_d_transactions.parquet')

In [6]:
# data inspection after Parquet conversion
# create a interim loader instance
interim_loader = SAMLDDataLoader(
    raw_data_path = raw_data_path,
    interim_data_path = interim_data_path
)

# create a lazy query plan
interim_lazy_frame = interim_loader.scan_interim()

# display LazyFrame schema
display(interim_lazy_frame.collect_schema())

# load a LazyFrame interim sample
interim_sample = interim_loader.load_interim_sample(
    n_rows = 10_000
)

# inspect first 10 rows
display(interim_sample.head(10))

# inspecting the primary label
display(
    interim_sample
    .group_by('is_laundering')
    .len()
)

Schema([('transaction_id', UInt64),
        ('timestamp', Datetime(time_unit='us', time_zone=None)),
        ('sender_account', String),
        ('receiver_account', String),
        ('amount', Float64),
        ('payment_currency', String),
        ('received_currency', String),
        ('sender_bank_location', String),
        ('receiver_bank_location', String),
        ('payment_type', String),
        ('is_laundering', Int8),
        ('laundering_type', String)])

transaction_id,timestamp,sender_account,receiver_account,amount,payment_currency,received_currency,sender_bank_location,receiver_bank_location,payment_type,is_laundering,laundering_type
u64,datetime[μs],str,str,f64,str,str,str,str,str,i8,str
0,2022-10-07 10:35:19,"""8724731955""","""2769355426""",1459.15,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,"""Normal_Cash_Deposits"""
1,2022-10-07 10:35:20,"""1491989064""","""8401255335""",6019.64,"""UK pounds""","""Dirham""","""UK""","""UAE""","""Cross-border""",0,"""Normal_Fan_Out"""
2,2022-10-07 10:35:20,"""287305149""","""4404767002""",14328.44,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,"""Normal_Small_Fan_Out"""
3,2022-10-07 10:35:21,"""5376652437""","""9600420220""",11895.0,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Fan_In"""
4,2022-10-07 10:35:21,"""9614186178""","""3803336972""",115.25,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cash Deposit""",0,"""Normal_Cash_Deposits"""
5,2022-10-07 10:35:21,"""8974559268""","""3143547511""",5130.99,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Group"""
6,2022-10-07 10:35:23,"""980191499""","""8577635959""",12176.52,"""UK pounds""","""UK pounds""","""UK""","""UK""","""ACH""",0,"""Normal_Small_Fan_Out"""
7,2022-10-07 10:35:23,"""8057793308""","""9350896213""",56.9,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Credit card""",0,"""Normal_Small_Fan_Out"""
8,2022-10-07 10:35:26,"""6116657264""","""656192169""",4738.45,"""UK pounds""","""UK pounds""","""UK""","""UK""","""Cheque""",0,"""Normal_Fan_Out"""


is_laundering,len
i8,u32
0,9987
1,13


In [7]:
### data validation after Parquet conversion
# validate row preservation and timestamp construction
interim_summary = (
    interim_lazy_frame
    .select(
        [
            pl.len()
            .alias(
                'row_count'
            ),
            pl.col('transaction_id')
            .n_unique()
            .alias(
                'unique_transaction_id'
            ),
            pl.col('timestamp')
            .null_count()
            .alias(
                'timestamp_null_count'
            ),
            pl.col('timestamp')
            .min()
            .alias(
                'minimum_timestamp'
            ),
            pl.col('timestamp')
            .max()
            .alias(
                'maximum_timestamp'
            )
        ]
    )
    .collect(
        engine = 'streaming'
    )
)

display(interim_summary)

# explicit row preservation 
interim_row_count = int(interim_summary['row_count'][0])
expected_row_count = int(raw_content_summary['row_count'])

if interim_row_count != expected_row_count:
    raise ValueError(
        'Raw-to-interim row count reconciliation failed. '
        f'Expected rows: {expected_row_count:,}, '
        f'Interim rows: {interim_row_count:,}.'
    )

print(
    'Raw-to-interim row count reconciliation passed. '
    f'Reconciled rows: {interim_row_count:,} rows'
)

row_count,unique_transaction_id,timestamp_null_count,minimum_timestamp,maximum_timestamp
u32,u32,u32,datetime[μs],datetime[μs]
9504852,9504852,0,2022-10-07 10:35:19,2023-08-23 10:57:12


Raw-to-interim row count reconciliation passed. Reconciled rows: 9,504,852 rows
